# Carga y Validación de Datos — `cargar_datos.ipynb`

**Proyecto:** Credit Risk MLOps Pipeline (Módulo 5)

**Propósito:** cargar la fuente de datos histórica de créditos (`base_de_datos.csv`) aplicando un conjunto de validaciones productivas antes de dejarla disponible para el análisis exploratorio (`comprension_eda.ipynb`).

**Validaciones implementadas:**
1. Existencia del archivo en la ruta esperada.
2. Encoding del archivo (UTF-8).
3. Presencia de todas las columnas esperadas (esquema de contrato).
4. Tipos de datos coherentes con el esquema esperado.
5. Manejo explícito de excepciones en cada etapa, con mensajes trazables para logs de Jenkins.

In [1]:
# ---------------------------------------------------------------
# 1. Importación de librerías
# ---------------------------------------------------------------
# Se importan únicamente las librerías necesarias para esta etapa:
# manejo de rutas, detección de encoding, y manipulación tabular.
from __future__ import annotations

import logging
from pathlib import Path
from typing import Final

import pandas as pd

# Configuración de logging: en un entorno productivo (Jenkins), este
# logger se propaga al log del job, permitiendo auditar cada corrida.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("cargar_datos")

In [2]:
# ---------------------------------------------------------------
# 2. Definición de excepciones propias
# ---------------------------------------------------------------
# Se definen excepciones específicas del dominio en lugar de usar
# excepciones genéricas: facilita el manejo diferenciado de errores
# aguas abajo (por ejemplo, en el pipeline de Jenkins) y deja logs
# más claros para auditoría.


class ArchivoNoEncontradoError(FileNotFoundError):
    """Se lanza cuando la fuente de datos no existe en la ruta esperada."""


class EncodingInvalidoError(ValueError):
    """Se lanza cuando el archivo no puede leerse como UTF-8."""


class EsquemaColumnasError(ValueError):
    """Se lanza cuando faltan columnas exigidas por el contrato de datos."""


class TipoDatoInvalidoError(TypeError):
    """Se lanza cuando una columna no cumple el tipo de dato esperado."""

In [3]:
# ---------------------------------------------------------------
# 3. Contrato de datos esperado (esquema)
# ---------------------------------------------------------------
# Este esquema es el "contrato" que Jenkins puede validar en cada
# ejecución del pipeline: si la fuente cambia de forma no controlada
# (columna renombrada, tipo cambiado), el pipeline debe fallar aquí,
# de forma temprana y explícita, en lugar de propagar datos corruptos
# a las etapas de modelamiento.

RUTA_DATOS: Final[Path] = Path("../base_de_datos.csv")

# Tipos lógicos esperados por columna (independientes del dtype exacto
# de pandas, que puede variar levemente entre versiones).
ESQUEMA_ESPERADO: Final[dict[str, str]] = {
    "tipo_credito": "numerico",
    "fecha_prestamo": "fecha",
    "capital_prestado": "numerico",
    "plazo_meses": "numerico",
    "edad_cliente": "numerico",
    "tipo_laboral": "categorico",
    "salario_cliente": "numerico",
    "total_otros_prestamos": "numerico",
    "cuota_pactada": "numerico",
    "puntaje": "numerico",
    "puntaje_datacredito": "numerico",
    "cant_creditosvigentes": "numerico",
    "huella_consulta": "numerico",
    "saldo_mora": "numerico",
    "saldo_total": "numerico",
    "saldo_principal": "numerico",
    "saldo_mora_codeudor": "numerico",
    "creditos_sectorFinanciero": "numerico",
    "creditos_sectorCooperativo": "numerico",
    "creditos_sectorReal": "numerico",
    "promedio_ingresos_datacredito": "numerico",
    "tendencia_ingresos": "categorico",
    "Pago_atiempo": "numerico",  # variable objetivo (0/1)
}

In [4]:
# ---------------------------------------------------------------
# 4. Funciones de validación
# ---------------------------------------------------------------


def validar_existencia_archivo(ruta: Path) -> None:
    """Valida que el archivo de datos exista en la ruta esperada.

    Args:
        ruta: Ruta del archivo a validar.

    Raises:
        ArchivoNoEncontradoError: Si el archivo no existe en disco.
    """
    if not ruta.exists():
        mensaje = f"No se encontró el archivo de datos en: {ruta}"
        logger.error(mensaje)
        raise ArchivoNoEncontradoError(mensaje)
    logger.info("Validación de existencia OK: %s", ruta)


def validar_encoding(ruta: Path, encoding_esperado: str = "utf-8") -> None:
    """Valida que el archivo pueda decodificarse con el encoding esperado.

    Se realiza una lectura binaria y un intento de decodificación explícito
    en lugar de delegar el error a `pandas.read_csv`, de forma que el mensaje
    de fallo sea inequívocamente atribuible a un problema de encoding.

    Args:
        ruta: Ruta del archivo a validar.
        encoding_esperado: Encoding contra el que se valida (por defecto
            UTF-8, estándar del proyecto).

    Raises:
        EncodingInvalidoError: Si el archivo no puede decodificarse con el
            encoding esperado.
    """
    try:
        with ruta.open("rb") as archivo_binario:
            contenido_binario = archivo_binario.read()
        contenido_binario.decode(encoding_esperado)
    except UnicodeDecodeError as error:
        mensaje = (
            f"El archivo {ruta} no está codificado en {encoding_esperado}: {error}"
        )
        logger.error(mensaje)
        raise EncodingInvalidoError(mensaje) from error
    logger.info("Validación de encoding OK: %s", encoding_esperado)


def validar_columnas(
    dataframe: pd.DataFrame, columnas_esperadas: list[str]
) -> None:
    """Valida que el DataFrame contenga todas las columnas del contrato.

    Args:
        dataframe: DataFrame cargado a validar.
        columnas_esperadas: Lista de nombres de columna exigidos por el
            contrato de datos.

    Raises:
        EsquemaColumnasError: Si falta alguna columna esperada.
    """
    columnas_faltantes = set(columnas_esperadas) - set(dataframe.columns)
    if columnas_faltantes:
        mensaje = (
            "Faltan columnas exigidas por el contrato de datos: "
            f"{sorted(columnas_faltantes)}"
        )
        logger.error(mensaje)
        raise EsquemaColumnasError(mensaje)
    logger.info("Validación de columnas OK: %d columnas esperadas presentes", len(columnas_esperadas))


def validar_tipos(
    dataframe: pd.DataFrame, esquema_esperado: dict[str, str]
) -> None:
    """Valida que cada columna sea compatible con el tipo lógico esperado.

    Args:
        dataframe: DataFrame cargado a validar.
        esquema_esperado: Diccionario columna -> tipo lógico ("numerico",
            "categorico" o "fecha").

    Raises:
        TipoDatoInvalidoError: Si alguna columna no es compatible con su
            tipo lógico esperado.
    """
    errores: list[str] = []
    for columna, tipo_logico in esquema_esperado.items():
        if columna not in dataframe.columns:
            continue  # ya reportado por validar_columnas
        serie = dataframe[columna]
        if tipo_logico == "numerico" and not pd.api.types.is_numeric_dtype(serie):
            errores.append(f"{columna}: se esperaba numérico, se obtuvo {serie.dtype}")
        elif tipo_logico == "fecha" and not pd.api.types.is_datetime64_any_dtype(serie):
            errores.append(f"{columna}: se esperaba fecha, se obtuvo {serie.dtype}")
        elif tipo_logico == "categorico" and not (
            pd.api.types.is_object_dtype(serie)
            or isinstance(serie.dtype, pd.CategoricalDtype)
            or pd.api.types.is_string_dtype(serie)
        ):
            errores.append(f"{columna}: se esperaba categórico, se obtuvo {serie.dtype}")
    if errores:
        mensaje = "Columnas con tipo de dato inválido: " + "; ".join(errores)
        logger.error(mensaje)
        raise TipoDatoInvalidoError(mensaje)
    logger.info("Validación de tipos OK: %d columnas verificadas", len(esquema_esperado))

In [5]:
# ---------------------------------------------------------------
# 5. Función orquestadora de carga
# ---------------------------------------------------------------


def cargar_datos(
    ruta: Path = RUTA_DATOS,
    esquema_esperado: dict[str, str] | None = None,
) -> pd.DataFrame:
    """Carga y valida la fuente de datos histórica de créditos.

    Ejecuta, en orden, las validaciones de existencia, encoding, columnas
    y tipos de dato, antes de retornar el DataFrame listo para EDA.

    Args:
        ruta: Ruta del archivo CSV de origen.
        esquema_esperado: Esquema columna -> tipo lógico. Si es ``None``,
            se usa `ESQUEMA_ESPERADO`.

    Returns:
        DataFrame de pandas validado, con `fecha_prestamo` parseada como
        fecha.

    Raises:
        ArchivoNoEncontradoError: Si el archivo no existe.
        EncodingInvalidoError: Si el encoding no es UTF-8.
        EsquemaColumnasError: Si faltan columnas del contrato.
        TipoDatoInvalidoError: Si algún tipo de dato no es compatible.
    """
    esquema_esperado = esquema_esperado or ESQUEMA_ESPERADO

    validar_existencia_archivo(ruta)
    validar_encoding(ruta)

    try:
        dataframe = pd.read_csv(
            ruta,
            encoding="utf-8",
            parse_dates=["fecha_prestamo"],
        )
    except pd.errors.ParserError as error:
        mensaje = f"Error de parseo al leer {ruta}: {error}"
        logger.error(mensaje)
        raise
    except pd.errors.EmptyDataError as error:
        mensaje = f"El archivo {ruta} está vacío: {error}"
        logger.error(mensaje)
        raise

    validar_columnas(dataframe, list(esquema_esperado.keys()))
    validar_tipos(dataframe, esquema_esperado)

    logger.info(
        "Carga completada con éxito: %d filas x %d columnas",
        dataframe.shape[0],
        dataframe.shape[1],
    )
    return dataframe

In [6]:
# ---------------------------------------------------------------
# 6. Ejecución
# ---------------------------------------------------------------
# Se envuelve la ejecución en un try/except adicional a nivel de notebook
# para simular el comportamiento que tendría este notebook al ser 
# orquestado como un job de Jenkins (captura de error de alto nivel, 
# antes de propagar el fallo al pipeline).
try:
    df_creditos = cargar_datos()
except (
    ArchivoNoEncontradoError,
    EncodingInvalidoError,
    EsquemaColumnasError,
    TipoDatoInvalidoError,
) as error:
    logger.critical("Fallo crítico en la carga de datos: %s", error)
    raise

df_creditos.shape

2026-07-28 22:02:18,027 | INFO | Validación de existencia OK: ../base_de_datos.csv


2026-07-28 22:02:18,042 | INFO | Validación de encoding OK: utf-8


2026-07-28 22:02:18,099 | INFO | Validación de columnas OK: 23 columnas esperadas presentes


2026-07-28 22:02:18,101 | INFO | Validación de tipos OK: 23 columnas verificadas


2026-07-28 22:02:18,102 | INFO | Carga completada con éxito: 10763 filas x 23 columnas


(10763, 23)

In [7]:
# Vista rápida del resultado de la carga
df_creditos.head()

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,3692160.0,10,42,Independiente,8000000,2500000,341296,88.768094,...,0.0,51258.0,51258.0,0.0,5,0,0,908526.0,Estable,1
1,4,2025-04-22 09:47:35,840000.0,6,60,Empleado,3000000,2000000,124876,95.227787,...,0.0,8673.0,8673.0,0.0,0,0,2,939017.0,Creciente,1
2,9,2026-01-08 12:22:40,5974028.4,10,36,Independiente,4036000,829000,529554,47.613894,...,0.0,18702.0,18702.0,0.0,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,1671240.0,6,48,Empleado,1524547,498000,252420,95.227787,...,0.0,15782.0,15782.0,0.0,3,0,0,1536193.0,Creciente,1
4,9,2025-04-26 11:24:26,2781636.0,11,44,Empleado,5000000,4000000,217037,95.227787,...,0.0,204804.0,204804.0,0.0,3,0,1,933473.0,Creciente,1


In [8]:
# Tipos de dato inferidos por pandas al cargar el archivo
df_creditos.dtypes

tipo_credito                              int64
fecha_prestamo                   datetime64[us]
capital_prestado                        float64
plazo_meses                               int64
edad_cliente                              int64
tipo_laboral                                str
salario_cliente                           int64
total_otros_prestamos                     int64
cuota_pactada                             int64
puntaje                                 float64
puntaje_datacredito                     float64
cant_creditosvigentes                     int64
huella_consulta                           int64
saldo_mora                              float64
saldo_total                             float64
saldo_principal                         float64
saldo_mora_codeudor                     float64
creditos_sectorFinanciero                 int64
creditos_sectorCooperativo                int64
creditos_sectorReal                       int64
promedio_ingresos_datacredito           

## Conclusiones de la etapa de carga

- El archivo `base_de_datos.csv` existe en la ruta esperada, está codificado en UTF-8 y se cargó sin errores de parseo.
- El DataFrame resultante tiene **10.763 filas y 23 columnas**, coincidiendo con el contrato de datos definido en `ESQUEMA_ESPERADO`.
- Las 23 columnas requeridas por el contrato están presentes y todas superaron la validación de tipo lógico (numérico/categórico/fecha).
- `fecha_prestamo` se parseó correctamente como tipo fecha (`datetime64`).
- **No se realizó limpieza de datos en este notebook a propósito**: la responsabilidad de `cargar_datos.ipynb` es exclusivamente cargar y validar el contrato estructural. La limpieza, el tratamiento de nulos y de valores atípicos se aborda en `comprension_eda.ipynb`, manteniendo una separación clara de responsabilidades entre etapas del pipeline.
- Próximo paso: pasar `df_creditos` (o releer `base_de_datos.csv`) al notebook `comprension_eda.ipynb` para el análisis exploratorio completo.